In [17]:
from pathlib import Path
import os

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_transformers import EmbeddingsRedundantFilter, LongContextReorder
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.merger_retriever import MergerRetriever

load_dotenv()

# These values mirror the pattern used in src_3/config.py and src_3/llm.py.
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.2"))
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "500"))
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_PERSIST_DIR = Path("./chroma_store").resolve()

print("Using chat model:", CHAT_MODEL)
print("Using embedding model:", EMBEDDING_MODEL)
print("Chroma persistence directory:", CHROMA_PERSIST_DIR)


Using chat model: gpt-5.4-nano
Using embedding model: text-embedding-3-small
Chroma persistence directory: D:\2027\Projects\RAG\Lessons\chroma_store


In [18]:
# Step 1: Create sample documents that simulate two knowledge sources.
# This version intentionally includes near-duplicate documents so the redundancy filter has something visible to remove.
hp_docs = [
    Document(page_content="Harry Potter is a wizard who attends Hogwarts School."),
    Document(page_content="Harry Potter is a student at Hogwarts School of Witchcraft and Wizardry."),
    Document(page_content="Harry's parents were killed by the dark wizard Voldemort."),
    Document(page_content="Hermione Granger is Harry's best friend and a brilliant witch."),
    Document(page_content="Ron Weasley is also Harry's close friend from Gryffindor."),
    Document(page_content="Hogwarts has four houses: Gryffindor, Slytherin, Ravenclaw, and Hufflepuff."),
]

# Game of Thrones Database
got_docs = [
    Document(page_content="Jon Snow is a central character in Game of Thrones."),
    Document(page_content="Jon Snow is an important character in the story Game of Thrones."),
    Document(page_content="Jon Snow was raised as a bastard but is actually a Targaryen."),
    Document(page_content="Daenerys Targaryen is the Mother of Dragons."),
    Document(page_content="The Seven Kingdoms are ruled by various noble houses."),
    Document(page_content="The Wall protects the realm from White Walkers beyond it."),
]

print("Created Harry Potter documents:", len(hp_docs))
print("Created Game of Thrones documents:", len(got_docs))
print("Note: some documents are intentionally similar so the redundancy filter can remove them.")


Created Harry Potter documents: 6
Created Game of Thrones documents: 6
Note: some documents are intentionally similar so the redundancy filter can remove them.


In [23]:
# Step 4: Create embeddings and initialize a Chroma retriever.
# This replaces the simple in-memory retriever with a real vector-store-based retriever.
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=OPENAI_API_KEY,
)

print("Embedding model ready:", EMBEDDING_MODEL)

# Create a local Chroma collection from the sample documents.
chroma_client = Chroma(
    collection_name="merge_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PERSIST_DIR),
)

# Clear any old data from the previous run so the demo starts fresh.
try:
    chroma_client.delete_collection()
    print("Cleared existing Chroma collection.")
except Exception as e:
    print("No existing collection to clear or delete failed:", e)

# Recreate the collection and add the sample documents.
chroma_client = Chroma(
    collection_name="merge_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PERSIST_DIR),
)
chroma_client.add_documents(hp_docs + got_docs)

# Build a Chroma-based retriever.
chroma_retriever = chroma_client.as_retriever(search_kwargs={"k": 3})
print("Chroma retriever ready for semantic search.")


Embedding model ready: text-embedding-3-small
Cleared existing Chroma collection.
Chroma retriever ready for semantic search.


In [24]:
# Step 5: Build a compact compression pipeline.
filter = EmbeddingsRedundantFilter(
    embeddings=embeddings,
    similarity_threshold=0.90,
)
reordering = LongContextReorder()
pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])

print("Compression pipeline ready")
print("Similarity threshold:", filter.similarity_threshold)


Compression pipeline created with:
- EmbeddingsRedundantFilter
- LongContextReorder
Similarity threshold: 0.9


In [25]:
# Step 6: Wrap the retriever in a contextual compression retriever.
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline,
    base_retriever=chroma_retriever,
)

print("ContextualCompressionRetriever created successfully.")


ContextualCompressionRetriever created successfully.


In [26]:
# Step 7: Run the final retrieval flow with the compression pipeline.
query = "Who is Jon Snow?"
raw_results = chroma_retriever.invoke(query)

print("Before compression - Stage 1: Raw results from Chroma")
for i, doc in enumerate(raw_results, 1):
    print(f"{i}. {doc.page_content}")

filtered_docs = filter.transform_documents(raw_results)
print("\nAfter redundancy filtering - Stage 2")
for i, doc in enumerate(filtered_docs, 1):
    print(f"{i}. {doc.page_content}")

reordered_docs = reordering.transform_documents(filtered_docs)
print("\nAfter reordering - Stage 3")
for i, doc in enumerate(reordered_docs, 1):
    print(f"{i}. {doc.page_content}")

filtered_results = compression_retriever.invoke(query)
print("\nAfter full compression pipeline - Stage 4")
for i, doc in enumerate(filtered_results, 1):
    print(f"{i}. {doc.page_content}")

print("\nSummary:")
print("Raw results:", len(raw_results))
print("After redundancy filter:", len(filtered_docs))
print("After reordering:", len(reordered_docs))
print("After full compression pipeline:", len(filtered_results))


Stage 1 - Raw results from Chroma:
1. Jon Snow is a central character in Game of Thrones.
2. Jon Snow is an important character in the story Game of Thrones.
3. Jon Snow was raised as a bastard but is actually a Targaryen.

Stage 2 - After redundancy filtering:
1. Jon Snow is an important character in the story Game of Thrones.
2. Jon Snow was raised as a bastard but is actually a Targaryen.

Stage 3 - After reordering for better context:
1. Jon Snow was raised as a bastard but is actually a Targaryen.
2. Jon Snow is an important character in the story Game of Thrones.

Stage 4 - Final results used by the compression retriever:
1. Jon Snow was raised as a bastard but is actually a Targaryen.
2. Jon Snow is an important character in the story Game of Thrones.

Summary:
Raw results: 3
After redundancy filter: 2
After reordering: 2
Final compressed results: 2

Why you should see a change:
The first two documents are intentionally similar, so the redundancy filter removes one of them.


In [27]:
# Step 8: Show the direct Chroma vector results for comparison.
vector_results = chroma_retriever.invoke("Who is Harry Potter?")
print("Chroma vector results:")
for i, doc in enumerate(vector_results, 1):
    print(f"{i}. {doc.page_content}")


Chroma vector results:
1. Harry Potter is a student at Hogwarts School of Witchcraft and Wizardry.
2. Harry Potter is a wizard who attends Hogwarts School.
3. Hermione Granger is Harry's best friend and a brilliant witch.


In [29]:
# Step 9: Build the final answer with an LLM using the retrieved context.
llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    api_key=OPENAI_API_KEY,
)

query = "Summarize the main idea about Harry Potter and Jon Snow."
context_docs = filtered_results[:3]
context_text = "\n\n".join(doc.page_content for doc in context_docs)

prompt = f"""You are a helpful assistant.
Use the following retrieved context to answer the question.

Context:
{context_text}

Question:
{query}
"""

response = llm.invoke(prompt)
print(response.content)

print("\nToken usage metadata:")
print(response.response_metadata.get("token_usage"))


Both **Harry Potter** and **Jon Snow** are central characters whose stories focus on identity and belonging: each is seen by others as an “outsider” or lesser figure based on how they were raised, but they ultimately have much deeper, more significant ties to an important lineage or destiny.

Token usage metadata:
{'completion_tokens': 62, 'prompt_tokens': 68, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}
